In [7]:
from __future__ import annotations

import sys
from copy import deepcopy
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [8]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.models.ebm import EnergyModel
from ebm_unlearning.src.training.pretrain import load_pretrained
from ebm_unlearning.src.training.unlearn import UnlearnConfig, unlearn
from ebm_unlearning.src.utils.logging import setup_logger
from ebm_unlearning.src.utils.seed import set_seed
from ebm_unlearning.src.utils.tracking import make_tracker


In [9]:
with open(ROOT / "configs" / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

set_seed(int(cfg["seed"]))

from datetime import datetime

device = torch.device(cfg.get("device", "cpu"))
logger = setup_logger("unlearn", log_file=str(ROOT / "outputs" / "logs" / "unlearn.log"))
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
tracker = make_tracker(
    "tensorboard",
    log_dir=str(ROOT / "outputs" / "tensorboard" / cfg["data"]["dataset"] / "unlearn" / run_id),
)

# Data: build forget/retain loaders (train only for the update loop)
spec = DatasetSpec(name=cfg["data"]["dataset"], data_dir=str(ROOT / cfg["data"]["data_dir"]), train=True, download=True)
dset = load_dataset(spec)

forget_spec = ForgetSpec(mode=cfg["data"]["forget"]["mode"], class_label=cfg["data"]["forget"]["class_label"])
retain_spec = RetainSpec(mode=cfg["data"]["retain"]["mode"])
forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

batch_size = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])
forget_loader = DataLoader(forget_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
retain_loader = DataLoader(retain_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)

forget_holdout_loader = DataLoader(forget_holdout, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)
retain_holdout_loader = DataLoader(retain_holdout, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)


[data] loading cifar10 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data
Files already downloaded and verified


In [ ]:

# Models: load pretrained E0 and initialize E from it
E0 = EnergyModel(
    in_channels=int(cfg["model"]["in_channels"]),
    hidden_dim=int(cfg["model"]["hidden_dim"]),
    num_classes=int(cfg["model"].get("num_classes", 10)),
    embed_dim=int(cfg["model"].get("embed_dim", 128)),
    backbone=str(cfg["model"].get("backbone", "conv")),
    finetune_stages=int(cfg["model"].get("finetune_stages", 1)),
    imagenet_pretrained=bool(cfg["model"].get("imagenet_pretrained", True)),
)
E0 = load_pretrained(E0, str(ROOT / cfg["pretrain"]["checkpoint_path"]), device=device)

E = deepcopy(E0)

un_cfg = UnlearnConfig(
    steps=int(cfg["unlearning"]["steps"]),
    lr=float(cfg["unlearning"]["lr"]),
    weight_decay=float(cfg["unlearning"]["weight_decay"]),
    lambda_f=float(cfg["unlearning"]["lambda_f"]),
    lambda_r=float(cfg["unlearning"]["lambda_r"]),
    lambda_m=float(cfg["unlearning"]["lambda_m"]),
    lambda_e=float(cfg["unlearning"]["lambda_e"]),
    margin=float(cfg["unlearning"]["margin"]),
    log_every=int(cfg["unlearning"]["log_every"]),
    checkpoint_path=str(ROOT / cfg["unlearning"]["checkpoint_path"]),
)

/home/owais/machine unlearning/ebm_unlearning/src/training/pretrain.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=devi

In [11]:
E = unlearn(
    E,
    E0,
    forget_loader,
    retain_loader,
    device=device,
    cfg=un_cfg,
    logger=logger,
    tracker=tracker,
    seed=int(cfg["seed"]),
    forget_holdout_loader=forget_holdout_loader,
    retain_holdout_loader=retain_holdout_loader,
)
tracker.close()

[2026-01-06 18:37:19,137] [INFO] [unlearn] step=0 margin=5.0000 gap_fw=-2.9068 total=12.729492 forget=8.054777 retain=0.000000 margin_loss=4.661821 energy_reg=12.893342
[2026-01-06 18:37:25,296] [INFO] [unlearn] step=50 margin=5.0000 gap_fw=2.2864 total=4.244696 forget=2.908760 retain=0.040591 margin_loss=0.918934 energy_reg=11.090797
[2026-01-06 18:37:31,843] [INFO] [unlearn] step=100 margin=5.0000 gap_fw=7.4637 total=0.779746 forget=0.134986 retain=0.056286 margin_loss=0.046530 energy_reg=35.374554
[2026-01-06 18:37:38,275] [INFO] [unlearn] step=150 margin=5.0000 gap_fw=7.9291 total=0.446154 forget=0.124990 retain=0.026585 margin_loss=0.014200 energy_reg=41.115570
[2026-01-06 18:37:44,650] [INFO] [unlearn] step=200 margin=5.0000 gap_fw=8.4652 total=0.335295 forget=0.047996 retain=0.023234 margin_loss=0.009650 energy_reg=45.305355
[2026-01-06 18:37:51,035] [INFO] [unlearn] step=250 margin=5.0000 gap_fw=9.0314 total=0.285391 forget=0.048414 retain=0.017718 margin_loss=0.012118 energy_r